# Data Cleaning Notebook (Phase III)
##### INFO 2950 Project by Anderson Ramirez (ar2527) and Jackson Rhamy (jtr84)
##### GitHub Repository: [View Project Repository](https://github.com/RamirezDevOps/info2950_data_project)

Our project asks whether Joe Biden's presidential approval rating is related to how often he is mentioned in political podcasts, and whether that relationship differs between liberal and conservative podcasts. This notebook prepares the data for that analysis. It combines two sources:

- Gallup's monthly presidential approval ratings for Joe Biden
- The Brookings Institution's Popular Political Podcast Dataset, which lists episode titles, descriptions, dates, and each show's political leaning

The raw files are in our [Google Drive folder](https://drive.google.com/drive/folders/1Qq1Ne2AHGYFrRtM5lEPPehE-ikWU0Dv6?usp=drive_link). This notebook cleans each dataset, identifies which podcast episodes mention Joe Biden, summarizes the episodes by month, and merges them with the approval ratings. It saves the cleaned files to `data/clean/` (listed at the end), and our main Phase III report loads them directly.

Changes since Phase II, based on TA feedback: episodes that mention both Hunter and Joe Biden now count as Joe Biden mentions, Hunter-only episodes are flagged and saved for review, every mention rate is also calculated with Hunter-only episodes counted, and we save a show-by-month table so each show can be weighted equally.

#### Package Imports

We use pandas and NumPy to clean and reshape the data, and DuckDB to run SQL queries for grouping and merging.

In [1]:
import pandas as pd
import numpy as np
import duckdb

### Data Collection and Cleaning

#### President Biden's Approval Ratings

The Where and What of this dataset:
- Gallup Presidential Approval Ratings: Joe Biden, provided by Gallup.
- The dataset can be downloaded as a CSV from the following link: https://news.gallup.com/poll/329384/presidential-approval-ratings-joe-biden.aspx

Why this dataset:
- This dataset provides information about Joe Biden’s presidential approval rating over time, allowing us to examine how his approval fluctuated during his presidency and compare those changes with the frequency of his mentions in political podcasts.

##### Loading President Biden's Approval Ratings CSV Data

Here we are loading the data, printing its shape and DataFrame.

In [2]:
# Load the CSV file containing Biden's approval ratings into a Pandas DataFrame
biden_ratings_df = pd.read_csv("../data/raw/bidenApprovalRating.csv")

print(f"The shape of biden_ratings_df is {biden_ratings_df.shape}")

biden_ratings_df.head()

The shape of biden_ratings_df is (56, 4)


,X.1,Approve,Disapprove,No opinion
0,NaN,%,%,%
1,<b>2025</b>,NaN,NaN,NaN
2,2025 Jan 2-15,40,54,6
3,<b>2024</b>,NaN,NaN,NaN
4,2024 Dec 2-18,39,56,5


##### Remove Headers, Unnecessary Columns, Rename Columns, and Convert Values

From the printed DataFrame, we can see that some rows have NaN values. These rows serve as headers that organize the data. For example, the first row specifies the units, in this case percentages, for the corresponding columns, while `<b>2025</b>` is a heading for the data collected in 2025.

Since these rows only provide additional information about the data rather than actual observations, we will remove the rows with NaNs. We will also remove the `Disapprove` and `No opinion` columns because our research questions focus on the approval rating specifically.

Next, we will rename the columns to make their labels more descriptive, and convert the percentage columns from an object to numeric data types to make sure they can be used for analysis. Since these values represent percentages, we will divide them by 100 to express them as a proportion between 0 and 1. This way the dataset will only have relevant observations with clear variable names and appropriate data types.

In [3]:
# Remove any rows with missing data and reset the row numbers to start from 0
biden_ratings_df = biden_ratings_df.dropna().reset_index(drop=True)

# Drop the columns we don't need
biden_ratings_df = biden_ratings_df.drop(columns=['Disapprove', 'No opinion'])

# Rename the columns to clearer names
biden_ratings_df = biden_ratings_df.rename(columns={'X.1':'Date', 'Approve':'Approval_Proportion'})

# Ensure the approval column is treated as decimal numbers
biden_ratings_df = biden_ratings_df.astype({'Approval_Proportion':float})

# Convert the approval percentages into true proportions
biden_ratings_df['Approval_Proportion'] = biden_ratings_df['Approval_Proportion'] / 100

biden_ratings_df.head()

,Date,Approval_Proportion
0,2025 Jan 2-15,0.40
1,2024 Dec 2-18,0.39
2,2024 Nov 6-20,0.37
3,2024 Oct 14-27,0.41
4,2024 Oct 1-12,0.39


In [4]:
# Creating intermediate saving point.
biden_ratings_df.to_csv('../data/clean/bidenApprovalRatingClean.csv', index=False)

##### Format the Date column

From the original printed DataFrame, we see that the dates are not consistently formatted, some months have duplicate observations, and two polling periods span multiple months. For example, October 2024 has two observations of .41 and .39, while all dates have polling ranges, like 2024 Oct 14-27, rather than standard monthly dates like 2024-10. Additionally, two polling periods cross month boundaries, 2022 Nov 9-Dec 2 and 2021 Jan 21-Feb 2.

- The first date issue we will tackle is the polling periods that span two months. To handle these cases, we will assign each polling observation to the month containing the midpoint of its polling period. This is consistent with the methodology used by Wlezien et al. (2013), who use the midpoint between a survey's fieldwork start and end dates to find its associated date (see References section for more information). Rather than arbitrarily assigning a survey to its start or end date, this method uses the midpoint of the actual data collection period. In practice, polls that fall entirely within one month are assigned using their start date, which is always in the same month as their midpoint. We set the two cross-month polls by hand to their midpoint dates: 2021 Jan 21-Feb 2 becomes January 27, 2021 (later removed as a partial month), and 2022 Nov 9-Dec 2 becomes November 20, 2022.
    - Because the Nov 9-Dec 2 poll is assigned to November, December 2022 has no approval observation. To keep a consistent monthly time series, we added a row for December 2022 with a missing approval proportion (`NaN`). We chose to preserve this missing value rather than estimate or interpolate it because doing so would introduce an approval proportion that was not directly observed in our original dataset. This approach follows the time-series data cleaning practices discussed in Lecture 7, which emphasize maintaining consistent time intervals, explicitly representing missing observations, and documenting how missing data is handled.

- Next, we will address the presence of multiple observations within the same month. To resolve this, we will convert the Date column object type to a Datetime type, and then find the average approval rating for each month that has multiple observations, using temporal aggregation, which combines higher frequency observations into a lower frequency, monthly dataset. Each poll is weighted equally regardless of its sample size or methodology; see the limitations section of our main report for more information. We chose this approach because our goal is to obtain one approval value per month for comparison with our podcast data, rather than develop a more sophisticated polling model.

- Finally, we are removing January 2021 and January 2025 from our data because Joe Biden was not president for the full month in either case. Since he took office on January 20, 2021, and left on January 20, 2025, these two months only offer a few days of data instead of a complete 30 or 31 days. Because our research compares monthly presidential approval ratings against monthly podcast mentions, every data point needs to look at an identical timeframe. Keeping these partial months could skew the results and make the comparison unfair.

As a result, our cleaned dataset contains 47 complete, consistent months running from February 2021 through December 2024, of which 46 have recorded approval proportions. Analyses requiring approval proportions will use the 46 nonmissing observations.

In [5]:
# Loading intermediate data
biden_ratings_df = pd.read_csv('../data/clean/bidenApprovalRatingClean.csv')

# Replace specific cross-month date ranges with representative single dates for consistency
biden_ratings_df['Date'] = biden_ratings_df['Date'].replace({'2021 Jan 21-Feb 2':'2021 Jan 27', '2022 Nov 9-Dec 2':'2022 Nov 20'})

# Add a missing December 2022 observation with a NaN approval proportion
biden_ratings_df = pd.concat([biden_ratings_df, pd.DataFrame({'Date': ['2022 Dec 20'], 'Approval_Proportion':[np.nan]})], ignore_index=True)

# Convert dates to datetime format and standardize them to the first day of each month
biden_ratings_df['Date'] = pd.to_datetime(biden_ratings_df['Date'].str.split('-').str[0]).dt.to_period('M').dt.to_timestamp()

# Group observations by month and calculate the average approval proportion
# Sort the results chronologically by date
biden_ratings_df = duckdb.sql('SELECT Date, AVG("Approval_Proportion") AS "Approval_Proportion", \
                                            FROM biden_ratings_df GROUP BY Date ORDER BY Date').df()

# Exclude partial months
biden_ratings_df = biden_ratings_df[biden_ratings_df['Date'].between('2021-02-01', '2024-12-01')].reset_index(drop=True)

print(f"Months: {len(biden_ratings_df)}, months missing approval: {biden_ratings_df['Approval_Proportion'].isna().sum()}")
biden_ratings_df.head()

Months: 47, months missing approval: 1


,Date,Approval_Proportion
0,2021-02-01,0.56
1,2021-03-01,0.54
2,2021-04-01,0.57
3,2021-05-01,0.54
4,2021-06-01,0.56


In [6]:
# Saving biden_ratings_df to a csv
biden_ratings_df.to_csv('../data/clean/bidenApprovalRatingClean.csv', index=False)

#### Political Podcast Data

The Where and What of this dataset:
- The dataset comes from the Brookings Institution’s Popular Political Podcast Dataset, which provides downloadable CSV data.
- It contains metadata for episodes of popular political podcasts, including show names, publication dates, titles, descriptions, and assigned political leanings.
- The dataset can be downloaded from Brookings at https://politicalpodcastproject.shinyapps.io/dataset/. Brookings updates it daily, so a new download will include more episodes than ours. Our copy has 94,855 rows, with episodes through November 2025, and is saved in our Google Drive folder. Because we keep only February 2021 through December 2024, later additions do not affect our analysis.

Why this dataset:
- We use this dataset to measure how frequently Joe Biden is mentioned in political podcast episode metadata over time and examine how mention rates vary by political leaning, allowing us to compare these patterns with his presidential approval ratings.

##### Loading Political Podcast Data

Before cleaning our podcast dataset, we need to examine its structure to understand what information is available and find any potential issues that could affect our analysis. We will start by inspecting the dimensions, column names, data types, and a sample of observations. This way we can see which columns are relevant and what cleaning steps are needed.

In [7]:
# Load the full podcast dataset from a CSV
podcasts_df = pd.read_csv("../data/raw/podcast_full-dataset.csv")

print(f"The shape of podcasts_df is {podcasts_df.shape}")

# Display column names, data types, and non-null counts
print(podcasts_df.info())

podcasts_df.head()

The shape of podcasts_df is (94855, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94855 entries, 0 to 94854
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         94855 non-null  int64 
 1   Episode ID         94855 non-null  int64 
 2   Show Name          94855 non-null  object
 3   Full Date          94855 non-null  object
 4   Title              94855 non-null  object
 5   Description        94831 non-null  object
 6   Partisan Leaning   94855 non-null  object
 7   Category on Apple  94855 non-null  object
 8   Entered Dataset    94855 non-null  object
 9   Audio URL          94852 non-null  object
dtypes: int64(2), object(8)
memory usage: 7.2+ MB
None


,Unnamed: 0,Episode ID,Show Name,Full Date,Title,Description,Partisan Leaning,Category on Apple,Entered Dataset,Audio URL
0,1,1,#SistersInLaw,2025-11-08,261: the earl of sandwich,jill wine-banks hosts #sistersinlaw to discuss...,More Liberal,News,Apple Recommendation,https://pdst.fm/e/arttrk.com/p/ABMA5/clrtpod.c...
1,2,2,#SistersInLaw,2025-11-01,"260: oh, snap",kimberly atkins stohr hosts #sistersinlaw to a...,More Liberal,News,Apple Recommendation,https://pdst.fm/e/arttrk.com/p/ABMA5/clrtpod.c...
2,3,3,#SistersInLaw,2025-10-25,"259: finally, an episode with sports",barb mcquade hosts #sistersinlaw to break down...,More Liberal,News,Apple Recommendation,https://pdst.fm/e/arttrk.com/p/ABMA5/clrtpod.c...
3,4,4,#SistersInLaw,2025-10-18,258: indictments for everyone,kimberly atkins stohr hosts #sistersinlaw to r...,More Liberal,News,Apple Recommendation,https://pdst.fm/e/arttrk.com/p/ABMA5/clrtpod.c...
4,5,5,#SistersInLaw,2025-10-11,257: unitary executive meets textualism,joyce vance hosts #sistersinlaw to analyze the...,More Liberal,News,Apple Recommendation,https://pdst.fm/e/arttrk.com/p/ABMA5/clrtpod.c...


##### Removing Unnecessary Columns and Handling Missing values

From the initial inspection, we found that multiple columns are not necessary for answering our research questions. Since our analysis focuses on the frequency of Biden mentions in podcast metadata, we will only keep `Show Name`, `Full Date`, `Title`, `Description`, and `Partisan Leaning`. The remaining columns will be dropped because they do not contribute to our analysis.

We also saw that the `Description` column is missing 24 values. Instead of removing these 24 episodes, we will replace these values with empty strings. This way we can keep the observations and still find mentions of Biden via the episode titles, even if they have a missing description.

In [8]:
# Remove unnecessary columns from the podcast dataset
podcasts_df = podcasts_df.drop(columns=['Episode ID', 'Unnamed: 0', 'Category on Apple', 'Entered Dataset', 'Audio URL'])

# Replace missing descriptions with empty strings
podcasts_df['Description'] = podcasts_df['Description'].fillna('')

print(f"The shape of podcasts_df is {podcasts_df.shape}")

# Check for remaining missing values in each column
print(f"Missing values after cleaning: \n{podcasts_df.isna().sum()}")

# Creating intermediate saving point.
podcasts_df.to_csv('../data/clean/podcastDataCleaned.csv', index=False)

podcasts_df.head()

The shape of podcasts_df is (94855, 5)
Missing values after cleaning: 
Show Name           0
Full Date           0
Title               0
Description         0
Partisan Leaning    0
dtype: int64


,Show Name,Full Date,Title,Description,Partisan Leaning
0,#SistersInLaw,2025-11-08,261: the earl of sandwich,jill wine-banks hosts #sistersinlaw to discuss...,More Liberal
1,#SistersInLaw,2025-11-01,"260: oh, snap",kimberly atkins stohr hosts #sistersinlaw to a...,More Liberal
2,#SistersInLaw,2025-10-25,"259: finally, an episode with sports",barb mcquade hosts #sistersinlaw to break down...,More Liberal
3,#SistersInLaw,2025-10-18,258: indictments for everyone,kimberly atkins stohr hosts #sistersinlaw to r...,More Liberal
4,#SistersInLaw,2025-10-11,257: unitary executive meets textualism,joyce vance hosts #sistersinlaw to analyze the...,More Liberal


##### Find Duplicates

We want to check if the dataset has any duplicate episodes. Duplicate observations can affect our results by causing certain episodes to be counted more than once, potentially inflating Biden's mentions.

We will check for completely identical rows, and then if any episodes share the same `Show Name`, `Full Date`, and `Title`, as this combination might show duplicate episodes.

In [9]:
print(f"Duplicate rows: {podcasts_df.duplicated().sum()}")

print(f"Potential duplicate episodes: {podcasts_df.duplicated(subset=['Show Name', 'Full Date', 'Title']).sum()}")

Duplicate rows: 144
Potential duplicate episodes: 192


The check found 144 completely identical rows and 192 potential duplicated episodes based on matching `Show Name`, `Full Date`, and `Title`. It is very possible that some episodes appear multiple times in the dataset. However, since the number of completely identical rows is lower, we will investigate these observations more before removing them.

In [10]:
# Find duplicate podcast episodes based on show name, date, and title
# Keep all occurrences of duplicates for inspection
duplicate_episodes = podcasts_df[podcasts_df.duplicated(subset=['Show Name', 'Full Date', 'Title'], keep=False)]

# Sort duplicate episodes by show name, date, and title
duplicate_episodes = duplicate_episodes.sort_values(by=['Show Name', 'Full Date', 'Title'])

duplicate_episodes.head(10)

,Show Name,Full Date,Title,Description,Partisan Leaning
2937,Bannon's War Room,2023-11-29,episode 3208: massive protest and unrest in ir...,episode 3208: massive protest and unrest in ir...,More Conservative
2938,Bannon's War Room,2023-11-29,episode 3208: massive protest and unrest in ir...,episode 3208: massive protest and unrest in ir...,More Conservative
1365,Bannon's War Room,2025-05-01,episode 4453: creating legislation to stop the...,episode 4453: creating legislation&nbsp;to sto...,More Conservative
1366,Bannon's War Room,2025-05-01,episode 4453: creating legislation to stop the...,episode 4453: creating legislation&nbsp;to sto...,More Conservative
9150,Bill O'Reilly's No Spin News and Analysis,2020-05-05,"the o'reilly update, may 5, 2020",the nation’s biggest supermarket chains begin ...,More Conservative
9151,Bill O'Reilly's No Spin News and Analysis,2020-05-05,"the o'reilly update, may 5, 2020","another day, another joe biden gaffe...learn m...",More Conservative
8939,Bill O'Reilly's No Spin News and Analysis,2020-08-19,"the o'reilly update, august 19, 2020","stocks recover losses caused by the contagion,...",More Conservative
8940,Bill O'Reilly's No Spin News and Analysis,2020-08-19,"the o'reilly update, august 19, 2020",undecided voters in america...learn more about...,More Conservative
8406,Bill O'Reilly's No Spin News and Analysis,2021-06-11,"o'reilly update morning edition, june 11, 2021","on this friday edition of the o'reilly update,...",More Conservative
8407,Bill O'Reilly's No Spin News and Analysis,2021-06-11,"o'reilly update morning edition, june 11, 2021","what is ""whiteness""...learn more about your ad...",More Conservative


##### Removing Duplicate Episodes

After seeing the potentially duplicated observations, we found that some episodes are completely identical across all five columns, while others share the same show, publication date, and title but have different descriptions.

Since descriptions are important to finding Biden mentions, we decided to only remove the 144 identical rows. We keep the 48 episodes (192 potential duplicates minus 144 exact duplicates) with different descriptions because they might represent different content, even if they share the same title and date.

In [11]:
# Remove duplicate rows and reset the DataFrame index
podcasts_df = podcasts_df.drop_duplicates().reset_index(drop=True)

# Check the number of remaining duplicate rows
print(f"Remaining duplicate rows: {podcasts_df.duplicated().sum()}")

print(f"The shape of podcasts_df now is {podcasts_df.shape}")

Remaining duplicate rows: 0
The shape of podcasts_df now is (94711, 5)


##### Filtering Episodes to Biden's Presidency

Because our research focuses on podcast episodes published during Biden's presidency, we will filter `podcasts_df` to include only episodes published from February 1, 2021, through December 31, 2024. This keeps our podcast data aligned with the 47 complete months isolated for our analysis.

In the initial inspection, we saw that `Full Date` is stored as an object rather than a datetime data type. Thus, we will convert this column into a Pandas datetime object to conduct some chronological filtering.

We will then filter `podcasts_df` directly to keep only episodes published during Biden's presidency, rather than making a separate DataFrame. Episodes outside this time frame will be excluded because they are not relevant to our research questions. We will also reset the index of the filtered DataFrame so that observations are numbered consecutively.

Finally, we will compare the dimensions of the dataset before and after filtering and make sure the publication dates match our desired time frame.

In [12]:
# Convert the 'Full Date' column to datetime format
podcasts_df['Full Date'] = pd.to_datetime(podcasts_df['Full Date'])

# Store the original dataset shape before filtering
og_shape = podcasts_df.shape

# Filter episodes to include only those published from February 01, 2021, through December 31, 2024
podcasts_df = podcasts_df[podcasts_df['Full Date'].between('2021-02-01', '2024-12-31')]

# Reset the DataFrame index after filtering
podcasts_df = podcasts_df.reset_index(drop=True)

# Display the original and filtered dataset dimensions
print(f"Original dataset shape: {og_shape}")
print(f"Filtered dataset shape: {podcasts_df.shape}")

# Check the earliest and latest publication dates in the filtered dataset
print(f"Earliest date: {podcasts_df['Full Date'].min()}")
print(f"Latest date: {podcasts_df['Full Date'].max()}")

# Creating intermediate saving point.
podcasts_df.to_csv('../data/clean/podcastDataCleaned.csv', index=False)

podcasts_df.head()

Original dataset shape: (94711, 5)
Filtered dataset shape: (56042, 5)
Earliest date: 2021-02-01 00:00:00
Latest date: 2024-12-31 00:00:00


,Show Name,Full Date,Title,Description,Partisan Leaning
0,#SistersInLaw,2024-12-28,216: super yacht rock,joyce vance hosts #sistersinlaw to review the ...,More Liberal
1,#SistersInLaw,2024-12-21,215: minnie mouse would do a better job,barb mcquade hosts #sistersinlaw to discuss fa...,More Liberal
2,#SistersInLaw,2024-12-14,214: misprision,jill wine-banks hosts #sistersinlaw to discuss...,More Liberal
3,#SistersInLaw,2024-12-07,213: pardon me,kimberly atkins stohr hosts #sistersinlaw to e...,More Liberal
4,#SistersInLaw,2024-11-30,212: keep calm and carry on,joyce vance hosts #sistersinlaw to investigate...,More Liberal


Filtering removed 38,669 episodes published outside Biden’s full months in office, leaving 56,042 episodes from February 1, 2021, through December 31, 2024.

##### Find Biden Mentions in Podcast Metadata

To find which podcast episodes mention Biden, we will parse through the Title and Description columns using Pandas `str.contains()` method.

First, we will check each column separately for the word "Biden," with case-insensitive matching to account for differences in capitalization and a regular expression to match the entire word. We will then combine the results from both columns to create a binary variable called `Biden_mention`, where 1 means Biden is in the title or description and 0 indicates he is not. Episodes where Biden is mentioned in both columns will only be counted once.

Because Hunter Biden was a massive presence in the general news cycle due to his criminal trial and legal troubles, his coverage could inflate our count of Joe Biden mentions. In Phase II, we set `Biden_mention` to 0 for any episode that mentioned Hunter Biden. Following TA feedback, we changed this rule because it also removed episodes that discussed both Hunter and Joe Biden:

- Episodes that mention both Hunter and Joe Biden keep the Joe Biden label. We check this by removing "Hunter Biden" from the text and seeing whether "Biden" still appears.
- Episodes that mention only Hunter Biden are flagged in a new `Hunter_only` column, set to 0 in `Biden_mention`, and saved to a separate file (`hunter_only_review.csv`) for review.
- We also keep `Biden_mention_incl_hunter`, which counts Hunter-only episodes as mentions, so we can compare our results when these episodes are counted and when they are not.

All episodes, including Hunter-only ones, remain in the total episode count; none are removed from the dataset. With this change, 13,360 episodes mention Joe Biden, 902 more than the 12,458 counted under the Phase II rule, and 554 episodes are flagged as Hunter-only.

In [13]:
# Find all mentions of "Biden" (whole word, any capitalization) in titles and descriptions
title_biden = podcasts_df['Title'].str.contains(r"\bBiden\b", case=False, na=False)
description_biden = podcasts_df['Description'].str.contains(r"\bBiden\b", case=False, na=False)

# Combine both columns: 1 if "Biden" appears in the title or description, 0 otherwise
podcasts_df['Biden_mention'] = (title_biden + description_biden).astype(int)

# Keep a copy that also counts Hunter-only episodes, for comparison
podcasts_df['Biden_mention_incl_hunter'] = podcasts_df['Biden_mention']

# Find episodes that mention "Hunter Biden" in the title or description
has_hunter_title = podcasts_df['Title'].str.contains("Hunter Biden", case=False, na=False)
has_hunter_desc = podcasts_df['Description'].str.contains("Hunter Biden", case=False, na=False)

# Check whether "Biden" still appears after removing "Hunter Biden" (i.e., the episode also refers to Joe Biden)
title_other = podcasts_df['Title'].str.replace("Hunter Biden", "", case=False, regex=True).str.contains(r"\bBiden\b", case=False)
desc_other = podcasts_df['Description'].str.replace("Hunter Biden", "", case=False, regex=True).str.contains(r"\bBiden\b", case=False)

# Flag Hunter-only episodes: they mention Hunter Biden and no other Biden
podcasts_df['Hunter_only'] = ((has_hunter_title | has_hunter_desc) & ~(title_other | desc_other)).astype(int)

# Set Hunter-only episodes to 0 in our main measure; episodes about both keep the Joe Biden label
podcasts_df['Biden_mention'] = np.where(podcasts_df['Hunter_only'] == 1, 0, podcasts_df['Biden_mention'])

print(podcasts_df["Biden_mention"].value_counts())

podcasts_df[["Title", "Description", "Biden_mention"]].head()

Biden_mention
0    42682
1    13360
Name: count, dtype: int64


,Title,Description,Biden_mention
0,216: super yacht rock,joyce vance hosts #sistersinlaw to review the ...,1
1,215: minnie mouse would do a better job,barb mcquade hosts #sistersinlaw to discuss fa...,0
2,214: misprision,jill wine-banks hosts #sistersinlaw to discuss...,1
3,213: pardon me,kimberly atkins stohr hosts #sistersinlaw to e...,1
4,212: keep calm and carry on,joyce vance hosts #sistersinlaw to investigate...,0


##### Review Hunter-Only Episodes

Following TA feedback, we save the episodes that mention only Hunter Biden to a separate file so they can be reviewed, rather than silently removing them. We also count how many of these episodes come from each political leaning, and look at the titles and descriptions of a random sample of 15 to check whether these episodes are really only about Hunter Biden, or whether some refer to Joe Biden indirectly (for example, as "the president" or "his father").

In [14]:
# Keep only the episodes flagged as Hunter-only
hunter_review = podcasts_df[podcasts_df['Hunter_only'] == 1]

# Save them to a separate file for review
hunter_review.to_csv('../data/clean/hunter_only_review.csv', index=False)

print(f"Hunter-only episodes: {len(hunter_review)}")

# How many Hunter-only episodes come from each political leaning
print(hunter_review['Partisan Leaning'].value_counts())

# Show a random sample of 15; random_state keeps the same sample every time we rerun
hunter_review[['Show Name', 'Partisan Leaning', 'Title', 'Description']].sample(15, random_state=2950)

Hunter-only episodes: 554
Partisan Leaning
More Conservative    406
More Liberal          76
Moderate              43
Unknown               29
Name: count, dtype: int64


,Show Name,Partisan Leaning,Title,Description
32657,The Clay Travis and Buck Sexton Show,More Conservative,hour 1 - hunter's capitol hill surprise,hunter biden makes surprise appearance at cong...
13465,Human Events Daily with Jack Posobiec,More Conservative,"jul 21, 2022 - riot justice: david dorn murder...",justice is served! a jury convicted the crimin...
2664,Bannon's War Room,More Conservative,warroom battleground ep 172: arizona has a ren...,warroom battleground ep 172: arizona has a ren...
49089,The News & Why It Matters,More Conservative,ep 737 | msm agenda: the real reason wapo refu...,the washington post makes a dramatic correctio...
30677,The Clay Travis and Buck Sexton Show,More Conservative,weekly review with clay and buck h3 - miranda ...,trump releases post calling for all hostages b...
27269,Rising,Moderate,'non-human biologics' recovered says ufo whist...,today: hunter biden to plead guilty to tax mis...
9547,Fast Politics with Molly Jong-Fast,More Liberal,"rick wilson, noah shachtman, michael liroff & ...",the lincoln project’s rick wilson roasts the g...
15389,John Solomon Reports,More Conservative,rep. andy biggs: officials who gave false test...,rep. andy biggs (r-az) discussed the 51 former...
11493,Firebrand with Matt Gaetz,More Conservative,episode 129 live: gaza disrupts congress – fir...,today on firebrand: congressman matt gaetz rec...
27796,Stay Tuned with Preet,More Liberal,cafe insider 8/22: georgia on my mind,"in this excerpt from the cafe insider podcast,..."


Of the 554 Hunter-only episodes, 406 (73%) come from conservative podcasts, 76 (14%) from liberal podcasts, 43 from moderate podcasts, and 29 from podcasts with an unknown leaning. Conservative podcasts make up about 61% of all episodes in our period and liberal podcasts about 23%, so Hunter-only coverage is concentrated in conservative podcasts more than their overall share would suggest.

To check whether these episodes are really only about Hunter Biden, we read the titles and descriptions of a random sample of 15. Five titles focus directly on Hunter Biden (for example, his laptop, his legal defense, and his business ties); in the other ten, he is one of several topics listed in the description. Thirteen of the 15 discuss Hunter Biden himself, such as his congressional appearance, plea deal, or trial. One refers to Joe Biden indirectly by discussing the timing of the Hunter Biden pardon, and one mentions Hunter Biden only in an advertisement for a film. This suggests that most Hunter-only episodes are not about Joe Biden, which supports leaving them out of our main measure, but a small number may be, so our main measure may slightly undercount Joe Biden mentions.

Because these episodes are concentrated in conservative podcasts, counting them as mentions would raise the conservative mention rate more than the liberal one. We therefore also report our results with Hunter-only episodes counted as mentions, especially for our comparison of liberal and conservative podcasts.

##### Inspect Political Leaning Column

We need to verify that our Partisan Leaning column is consistent because one of our research questions examines whether the relationship between Biden's approval rating and podcast mention frequency differs by political leaning.

We will see unique values in this column and count the number of observations belonging to each classification.

In [15]:
print(podcasts_df["Partisan Leaning"].unique())
print(podcasts_df["Partisan Leaning"].value_counts())

print(f"Missing values: {podcasts_df["Partisan Leaning"].isna().sum()}")

['More Liberal' 'More Conservative' 'Moderate' 'Unknown']
Partisan Leaning
More Conservative    33918
More Liberal         12865
Unknown               5362
Moderate              3897
Name: count, dtype: int64
Missing values: 0


The `Partisan Leaning` column has four categories, `More Liberal`, `More Conservative`, `Moderate`, and `Unknown`. There are no missing values, so no additional cleaning is necessary. We will keep the `Unknown` category to preserve all observations, but exclude it when comparing political leanings.

##### Grouping Episodes by Month

To find the relationship between Biden's monthly approval rating and his mention frequency in political podcasts, we will organize the dataset by month.

We will create a `Month` column using the `Full Date` column to identify the year and month each episode was published. Then, we will group the episodes by month and calculate the total number of episodes published and the number of episodes mentioning Biden. These values will help us calculate the monthly Biden mention rate as the proportion of episodes mentioning Biden out of all episodes published that month. These results will be stored in a separate DataFrame called `monthly_podcasts_df` while preserving the original episode data.

We also calculate a second monthly mention rate, `Mention_Proportion_InclHunter`, that counts Hunter-only episodes as mentions, so we can compare results when these episodes are counted and when they are not.

In [16]:
# Extract the year and month from each episode's publication date
podcasts_df["Month"] = podcasts_df["Full Date"].dt.to_period("M")

# Group episodes by month and calculate the total number of episodes and Biden mentions
monthly_podcasts_df = podcasts_df.groupby("Month").agg(
    Total_episodes=("Biden_mention", "count"),
    Biden_episodes=("Biden_mention", "sum"),
    Biden_episodes_incl_hunter=("Biden_mention_incl_hunter", "sum")
).reset_index()

# Calculate the monthly proportion of episodes mentioning Biden
monthly_podcasts_df["Mention_Proportion"] = (monthly_podcasts_df["Biden_episodes"] / monthly_podcasts_df["Total_episodes"])
monthly_podcasts_df["Mention_Proportion_InclHunter"] = (monthly_podcasts_df["Biden_episodes_incl_hunter"] / monthly_podcasts_df["Total_episodes"])

# Creating intermediate saving point.
podcasts_df.to_csv('../data/clean/podcastDataCleaned.csv', index=False)
monthly_podcasts_df.to_csv('../data/clean/monthlyPodcastData.csv', index=False)

monthly_podcasts_df.head()

,Month,Total_episodes,Biden_episodes,Biden_episodes_incl_hunter,Mention_Proportion,Mention_Proportion_InclHunter
0,2021-02,688,160,162,0.232558,0.235465
1,2021-03,802,247,248,0.307980,0.309227
2,2021-04,777,200,210,0.257400,0.270270
3,2021-05,740,158,159,0.213514,0.214865
4,2021-06,789,189,198,0.239544,0.250951


##### Show-by-Month Mention Data

Following TA feedback, we also calculate each show's own monthly mention proportion, so that later averages can give every show equal weight.

In [17]:
# Group episodes by show, leaning, and month
show_monthly = podcasts_df.groupby(["Show Name", "Partisan Leaning", "Month"]).agg(
    Episodes=("Biden_mention", "count"),
    Biden_episodes=("Biden_mention", "sum")
).reset_index()

# Each show's monthly proportion of episodes mentioning Biden
show_monthly["Show_Mention_Proportion"] = show_monthly["Biden_episodes"] / show_monthly["Episodes"]

show_monthly.to_csv('../data/clean/showMonthlyData.csv', index=False)

show_monthly.head()

,Show Name,Partisan Leaning,Month,Episodes,Biden_episodes,Show_Mention_Proportion
0,#SistersInLaw,More Liberal,2021-02,4,1,0.25
1,#SistersInLaw,More Liberal,2021-03,4,0,0.00
2,#SistersInLaw,More Liberal,2021-04,4,1,0.25
3,#SistersInLaw,More Liberal,2021-05,5,0,0.00
4,#SistersInLaw,More Liberal,2021-06,4,1,0.25


Each row of `show_monthly` is one show in one month. For example, #SistersInLaw (a liberal podcast) published 4 episodes in February 2021, and 1 of them mentioned Biden, giving a show-level mention proportion of 0.25. In March 2021, none of its 4 episodes mentioned Biden, so its proportion is 0.

Because each show gets its own monthly proportion, a show that publishes only a few episodes a month counts the same as a show that publishes dozens when we average across shows. This addresses the concern that a few frequently publishing shows could dominate the overall mention rates. We save this table to `showMonthlyData.csv` so the main analysis can calculate equal-weight averages and leave-one-show-out comparisons without additional data cleaning.

### Combined Monthly Dataset

The Where and What of this dataset:
- This is a derived dataset created by merging Gallup’s Biden presidential approval ratings with monthly statistics from the Brookings political podcast dataset, producing 47 monthly observations and 17 columns covering February 2021 through December 2024.

Why this dataset:
- We created this dataset to examine the relationship between Biden’s monthly presidential approval ratings and the proportion of political podcast episodes mentioning him, including how this relationship varies across liberal, moderate, and conservative podcasts.

##### Prepare the Datasets for Merging

Before merging the two datasets, we need to make sure the month columns use the same format and that their measurements are consistent.

In `biden_ratings_df`, the `Date` column represents the first day of each month, and `Approval_Proportion` is already expressed as a proportion between 0 and 1. In `monthly_podcasts_df`, `Month` is stored as a monthly period, while `Mention_Proportion` and `Mention_Proportion_InclHunter` are expressed as decimals ranging from 0 to 1.

We reload both monthly datasets from the CSV files we saved earlier, so this step does not depend on earlier variables still being in memory. Then, we will rename and convert the date columns into a common `Month` format. We will also recalculate the podcast mention rate as a proportion using the number of episodes that mention Biden divided by the total number of episodes, and keep only the columns needed for the merge.

Finally, we will conduct a sanity check for duplicate months in either dataset, since each month should only have one observation before we merge the data.

In [18]:
# Load Datasets
approval_monthly = pd.read_csv('../data/clean/bidenApprovalRatingClean.csv')
podcast_monthly = pd.read_csv('../data/clean/monthlyPodcastData.csv')

# Renames the approval date column and standardizes both month columns to the first day of each month.
approval_monthly = approval_monthly.rename(columns={"Date": "Month"})

approval_monthly['Month'] = pd.to_datetime(approval_monthly['Month']).dt.to_period('M').dt.to_timestamp()

podcast_monthly['Month'] = pd.to_datetime(podcast_monthly['Month'].astype(str)).dt.to_period('M').dt.to_timestamp()

# Recalculate the podcast mention proportion from counts
podcast_monthly["Mention_Proportion"] = podcast_monthly["Biden_episodes"] / podcast_monthly["Total_episodes"]

# Keep only columns needed
approval_monthly = approval_monthly[["Month", "Approval_Proportion"]]
podcast_monthly = podcast_monthly[["Month", "Total_episodes", "Biden_episodes", "Mention_Proportion", "Mention_Proportion_InclHunter"]]

# Verify there is one row per month
print(f"Duplicate approval months: {approval_monthly["Month"].duplicated().sum()}")
print(f"Duplicate podcast months: {podcast_monthly["Month"].duplicated().sum()}")

print(approval_monthly.head())
print(podcast_monthly.head())

Duplicate approval months: 0
Duplicate podcast months: 0
       Month  Approval_Proportion
0 2021-02-01                 0.56
1 2021-03-01                 0.54
2 2021-04-01                 0.57
3 2021-05-01                 0.54
4 2021-06-01                 0.56
       Month  Total_episodes  Biden_episodes  Mention_Proportion  \
0 2021-02-01             688             160            0.232558   
1 2021-03-01             802             247            0.307980   
2 2021-04-01             777             200            0.257400   
3 2021-05-01             740             158            0.213514   
4 2021-06-01             789             189            0.239544   

   Mention_Proportion_InclHunter  
0                       0.235465  
1                       0.309227  
2                       0.270270  
3                       0.214865  
4                       0.250951  


##### Merging Approval Ratings with Monthly Podcast Data

Now that both datasets have been standardized, we can combine them using the `Month` column. We use a left join with the podcast dataset as the primary table, which keeps all 47 months with podcast observations even if the corresponding approval proportion is missing (December 2022).

The result contains, for each month, the approval proportion, the total number of podcast episodes, the number of episodes mentioning Joe Biden, and the mention proportion both without (`Mention_Proportion`) and with (`Mention_Proportion_InclHunter`) Hunter-only episodes counted.

In [19]:
# Merge monthly podcast data with approval ratings, keeping every podcast month
combined_monthly_df = duckdb.sql('SELECT p.Month, p.Total_episodes, p.Biden_episodes, p.Mention_Proportion, p.Mention_Proportion_InclHunter, \
                                        a.Approval_Proportion FROM podcast_monthly AS p LEFT JOIN \
                                        approval_monthly AS a ON p.Month = a.Month ORDER BY p.Month').df()

print(f"Combined dataset shape: {combined_monthly_df.shape}")
combined_monthly_df.head()

Combined dataset shape: (47, 6)


,Month,Total_episodes,Biden_episodes,Mention_Proportion,Mention_Proportion_InclHunter,Approval_Proportion
0,2021-02-01,688,160,0.232558,0.235465,0.56
1,2021-03-01,802,247,0.307980,0.309227,0.54
2,2021-04-01,777,200,0.257400,0.270270,0.57
3,2021-05-01,740,158,0.213514,0.214865,0.54
4,2021-06-01,789,189,0.239544,0.250951,0.56


##### Mention Proportions by Political Leaning

To address our subquestion about political leaning, we use the cleaned episode-level podcast data to calculate monthly Biden mention proportions separately for conservative, moderate, liberal, and unknown-leaning podcasts. We then turn each leaning into its own column, so the combined dataset still has one row per month.

We also create `Liberal_Conservative_Difference`, the liberal mention proportion minus the conservative mention proportion (`Liberal_Mention` - `Conservative_Mention`). A positive value means liberal podcasts mentioned Biden more often that month, while a negative value means conservative podcasts mentioned him more often. This lets us examine how the gap between the two leanings changes over time and relates to Biden's approval rating.

In [20]:
# Load the cleaned episode-level podcast data
podcasts_leaning = pd.read_csv("../data/clean/podcastDataCleaned.csv")

# Standardize the Month column to the first day of each month
podcasts_leaning["Month"] = pd.to_datetime(podcasts_leaning["Month"]).dt.to_period("M").dt.to_timestamp()

# Group episodes by month and political leaning
leaning_monthly = podcasts_leaning.groupby(["Month", "Partisan Leaning"]).agg(
    Total_episodes=("Biden_mention", "count"),
    Biden_episodes=("Biden_mention", "sum"),
    Biden_episodes_incl_hunter=("Biden_mention_incl_hunter", "sum")
).reset_index()

# Calculate mention proportions for each political leaning
leaning_monthly["Mention_Proportion"] = leaning_monthly["Biden_episodes"] / leaning_monthly["Total_episodes"]

# Turn political leaning categories into separate columns
leaning_wide = leaning_monthly.pivot(index="Month", columns="Partisan Leaning", values="Mention_Proportion").reset_index()

# Rename the political leaning columns
leaning_wide = leaning_wide.rename(columns={
    "More Conservative": "Conservative_Mention",
    "Moderate": "Moderate_Mention",
    "More Liberal": "Liberal_Mention",
    "Unknown": "Unknown_Mention"
})

# Add political leaning mention proportions to the combined data
combined_monthly_df = combined_monthly_df.merge(leaning_wide, on='Month', how='left')

# Calculate the monthly difference between liberal and conservative mentions
combined_monthly_df["Liberal_Conservative_Difference"] = combined_monthly_df["Liberal_Mention"] - combined_monthly_df["Conservative_Mention"]

combined_monthly_df[["Month", "Conservative_Mention", "Liberal_Mention", "Liberal_Conservative_Difference"]].head()

,Month,Conservative_Mention,Liberal_Mention,Liberal_Conservative_Difference
0,2021-02-01,0.288889,0.197802,-0.091087
1,2021-03-01,0.397059,0.192308,-0.204751
2,2021-04-01,0.348028,0.178571,-0.169456
3,2021-05-01,0.275534,0.138614,-0.136921
4,2021-06-01,0.299776,0.190244,-0.109532


##### Leaning Proportions Including Hunter-Only Episodes

Following TA feedback, we repeat the leaning calculations with Hunter-only episodes counted as mentions. These columns end in `_InclHunter`, including `Liberal_Conservative_Difference_InclHunter`. Because Hunter-only episodes come mostly from conservative podcasts, this comparison matters most for the liberal-minus-conservative difference.

In [21]:
# Same leaning proportions, but counting Hunter-only episodes as mentions
leaning_monthly["Mention_Proportion_InclHunter"] = leaning_monthly["Biden_episodes_incl_hunter"] / leaning_monthly["Total_episodes"]

# Turn political leaning categories into separate columns
leaning_wide_hunter = leaning_monthly.pivot(index="Month", columns="Partisan Leaning", values="Mention_Proportion_InclHunter").reset_index()

# Rename the political leaning columns
leaning_wide_hunter = leaning_wide_hunter.rename(columns={
    "More Conservative": "Conservative_Mention_InclHunter",
    "Moderate": "Moderate_Mention_InclHunter",
    "More Liberal": "Liberal_Mention_InclHunter",
    "Unknown": "Unknown_Mention_InclHunter"
})

# Add the Hunter-included proportions to the combined data
combined_monthly_df = combined_monthly_df.merge(leaning_wide_hunter, on='Month', how='left')

# Calculate the Hunter-included liberal-minus-conservative difference
combined_monthly_df["Liberal_Conservative_Difference_InclHunter"] = combined_monthly_df["Liberal_Mention_InclHunter"] - combined_monthly_df["Conservative_Mention_InclHunter"]

combined_monthly_df[["Month", "Liberal_Conservative_Difference", "Liberal_Conservative_Difference_InclHunter"]].head()

,Month,Liberal_Conservative_Difference,Liberal_Conservative_Difference_InclHunter
0,2021-02-01,-0.091087,-0.096025
1,2021-03-01,-0.204751,-0.206852
2,2021-04-01,-0.169456,-0.176769
3,2021-05-01,-0.136921,-0.139296
4,2021-06-01,-0.109532,-0.129667


##### Month Counter and Final Checks

Our regressions include a month counter to account for changes over Biden's presidency. We create `Month_Number`, from 1 (February 2021) to 47 (December 2024), here in the cleaning notebook, before December 2022 is dropped for any analysis. This way the months after December 2022 keep their real position in time instead of shifting down by one.

Finally, we check the shape of the combined dataset and look for missing values. We expect 47 rows and 17 columns, with the only missing value being the December 2022 approval proportion.

In [22]:
# Month counter from 1 (Feb 2021) to 47 (Dec 2024), created before December 2022 is dropped so later months keep their real position in time
combined_monthly_df["Month_Number"] = range(1, len(combined_monthly_df) + 1)

# Check the shape of the merged dataset
print(f"Combined dataset shape: {combined_monthly_df.shape}")

# Check for missing values and display them
print(combined_monthly_df.isna().sum())
print(combined_monthly_df[combined_monthly_df.isna().any(axis=1)])

combined_monthly_df.head()

Combined dataset shape: (47, 17)
Month                                         0
Total_episodes                                0
Biden_episodes                                0
Mention_Proportion                            0
Mention_Proportion_InclHunter                 0
Approval_Proportion                           1
Moderate_Mention                              0
Conservative_Mention                          0
Liberal_Mention                               0
Unknown_Mention                               0
Liberal_Conservative_Difference               0
Moderate_Mention_InclHunter                   0
Conservative_Mention_InclHunter               0
Liberal_Mention_InclHunter                    0
Unknown_Mention_InclHunter                    0
Liberal_Conservative_Difference_InclHunter    0
Month_Number                                  0
dtype: int64
        Month  Total_episodes  Biden_episodes  Mention_Proportion  \
22 2022-12-01            1146             163            0.142234   


,Month,Total_episodes,Biden_episodes,Mention_Proportion,Mention_Proportion_InclHunter,Approval_Proportion,Moderate_Mention,Conservative_Mention,Liberal_Mention,Unknown_Mention,Liberal_Conservative_Difference,Moderate_Mention_InclHunter,Conservative_Mention_InclHunter,Liberal_Mention_InclHunter,Unknown_Mention_InclHunter,Liberal_Conservative_Difference_InclHunter,Month_Number
0,2021-02-01,688,160,0.232558,0.235465,0.56,0.142857,0.288889,0.197802,0.050000,-0.091087,0.142857,0.293827,0.197802,0.050000,-0.096025,1
1,2021-03-01,802,247,0.307980,0.309227,0.54,0.136364,0.397059,0.192308,0.156250,-0.204751,0.136364,0.399160,0.192308,0.156250,-0.206852,2
2,2021-04-01,777,200,0.257400,0.270270,0.57,0.136364,0.348028,0.178571,0.070000,-0.169456,0.136364,0.364269,0.187500,0.080000,-0.176769,3
3,2021-05-01,740,158,0.213514,0.214865,0.54,0.100000,0.275534,0.138614,0.123711,-0.136921,0.100000,0.277910,0.138614,0.123711,-0.139296,4
4,2021-06-01,789,189,0.239544,0.250951,0.56,0.081081,0.299776,0.190244,0.130000,-0.109532,0.081081,0.319911,0.190244,0.130000,-0.129667,5


In [23]:
# Creating intermediate saving point.
combined_monthly_df.to_csv('../data/clean/combinedMonthlyProportions.csv', index=False)

### Summary of Cleaned Files

| File | Contents | Used for |
|---|---|---|
| `bidenApprovalRatingClean.csv` | 47 months: `Date`, `Approval_Proportion` (December 2022 missing) | Merging |
| `podcastDataCleaned.csv` | 56,042 episodes with show, date, title, description, leaning, `Biden_mention`, `Biden_mention_incl_hunter`, `Hunter_only`, `Month` | Merging, review |
| `monthlyPodcastData.csv` | 47 months: total episodes, Biden episodes, and both monthly mention proportions | Merging |
| `hunter_only_review.csv` | The 554 Hunter-only episodes | Review of uncertain cases |
| `showMonthlyData.csv` | One row per show per month: episodes, Biden episodes, `Show_Mention_Proportion` | Equal-weight and leave-one-show-out checks |
| `combinedMonthlyProportions.csv` | 47 months × 17 columns (below) | Main analysis |

Columns in `combinedMonthlyProportions.csv` (all proportions are between 0 and 1):

| Column | Meaning |
|---|---|
| `Month` | First day of the month |
| `Total_episodes` | Episodes published that month |
| `Biden_episodes` | Episodes mentioning Joe Biden (main measure) |
| `Mention_Proportion` | `Biden_episodes / Total_episodes` |
| `Mention_Proportion_InclHunter` | Same, but counting Hunter-only episodes as mentions |
| `Approval_Proportion` | Average Gallup approval that month (missing for December 2022) |
| `Conservative_Mention`, `Liberal_Mention`, `Moderate_Mention`, `Unknown_Mention` | Mention proportion within each leaning |
| `Liberal_Conservative_Difference` | `Liberal_Mention − Conservative_Mention`; positive means liberal podcasts mentioned Biden more |
| Columns ending in `_InclHunter` | The same leaning proportions and difference, counting Hunter-only episodes as mentions |
| `Month_Number` | Month counter from 1 (February 2021) to 47 (December 2024); December 2022 is month 23 |

### AI Acknowledgment

We used Claude, a generative AI assistant, while revising this notebook for Phase III. Specifically, we used it to:

- Review our Phase II cleaning notebook against the Phase III/IV rubric and our TA's Phase II feedback, and list the changes needed.
- Draft and edit markdown explanations and code comments, remove excess bold formatting, split the merge step into smaller explained steps, and write the summary table of cleaned files.
- Summarize the random sample of Hunter-only episode descriptions, which we then checked against the notebook output.

We reviewed, and verified all text, checked every number against the notebook outputs, and made the final decisions about our cleaning approach. We take full responsibility for the content of this notebook.